In [0]:
# 02_silver_to_gold

storage_account_name = dbutils.secrets.get(scope="crypto-pipeline", key="storage-account-name")
storage_account_key  = dbutils.secrets.get(scope="crypto-pipeline", key="storage-account-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

# Read Silver Delta table
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/crypto/"

df_silver = spark.read.format("delta").load(silver_path)

print(f"✅ Loaded {df_silver.count()} records from Silver")

In [0]:
from pyspark.sql.functions import col, round, desc, sum as spark_sum

# Gold Table 1: Top 10 coins by market cap
df_top10 = df_silver \
    .select(
        "name",
        "symbol",
        "current_price",
        "market_cap",
        "market_cap_rank",
        "total_volume",
        "price_change_percentage_24h"
    ) \
    .orderBy("market_cap_rank") \
    .limit(10)

df_top10.show()

In [0]:
# Gold Table 2: Top 5 gainers and losers in 24h
df_gainers = df_silver \
    .select("name", "symbol", "current_price", "price_change_percentage_24h") \
    .orderBy(desc("price_change_percentage_24h")) \
    .limit(5)

df_losers = df_silver \
    .select("name", "symbol", "current_price", "price_change_percentage_24h") \
    .orderBy("price_change_percentage_24h") \
    .limit(5)

print("🟢 Top 5 Gainers:")
df_gainers.show()

print("🔴 Top 5 Losers:")
df_losers.show()

In [0]:
# Gold Table 3: Overall market summary
df_market_summary = df_silver.agg(
    spark_sum("market_cap").alias("total_market_cap"),
    spark_sum("total_volume").alias("total_volume"),
)

print("📊 Market Summary:")
df_market_summary.show()

In [0]:
# Write Gold tables to Azure Data Lake
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/crypto/"

# Top 10 by market cap
df_top10.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}top10_by_marketcap/")

# Gainers
df_gainers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}top5_gainers/")

# Losers
df_losers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}top5_losers/")

# Market summary
df_market_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}market_summary/")

print("✅ All Gold tables written successfully!")